# 🧠 AN-RA Training Notebook
**5 cells. Run top to bottom every session.**

| Cell | What it does |
|------|-------------|
| 1 | ⚙️ Config — only thing you ever edit |
| 2 | 🔧 Environment — GPU check + Drive mount + install deps |
| 3 | 📤 Repo — upload zip + apply fix + pre-flight check |
| 4 | 🚀 Run training |
| 5 | ✅ Verify Drive saves |


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║         AN-RA CONFIG — ONLY EDIT THIS CELL          ║
# ╚══════════════════════════════════════════════════════╝

SESSION_MINUTES    = 30   # ← change freely: 20, 30, 60, 90...
OUROBOROS_MINUTES  = 10   # ← Ouroboros recursive reasoning (45O)
OVERHEAD_MINUTES   = 5    # ← reserved for merge/audit/setup

# Training mode:
#   "train"   → full pipeline: identity + ouroboros + self-improvement + tests ✅
#   "session" → base transformer training only (faster, lighter)
TRAINING_MODE = "train"

# ── Derived (don't touch) ─────────────────────────────
REPO_PATH            = "/content/An-Ra-the-new-AGI"
DRIVE_ANRA_DIR       = "/content/drive/MyDrive/AnRa"
DRIVE_V2_CHECKPOINTS = "/content/drive/MyDrive/AnRa/v2/checkpoints"
DRIVE_DATASET        = "/content/drive/MyDrive/AnRa/anra_dataset_v6_1.txt"
IDENTITY_MINUTES     = max(1, SESSION_MINUTES - OVERHEAD_MINUTES - OUROBOROS_MINUTES)

print("╔══════════════════════════════════════════╗")
print("║         AN-RA SESSION PLAN               ║")
print("╠══════════════════════════════════════════╣")
print(f"║  Mode       : {TRAINING_MODE:<27}║")
print(f"║  Total      : {SESSION_MINUTES} min{' '*(25-len(str(SESSION_MINUTES)))}║")
print(f"║  Overhead   : {OVERHEAD_MINUTES} min{' '*(25-len(str(OVERHEAD_MINUTES)))}║")
print(f"║  Ouroboros  : {OUROBOROS_MINUTES} min{' '*(25-len(str(OUROBOROS_MINUTES)))}║")
print(f"║  Identity   : {IDENTITY_MINUTES} min (dynamic){' '*(14-len(str(IDENTITY_MINUTES)))}║")
print("╚══════════════════════════════════════════╝")


In [ ]:
# 🔧  ENVIRONMENT SETUP
# GPU diagnostics → Mount Drive → Install dependencies
import subprocess, sys, os, psutil, torch, glob
from google.colab import drive

# ── 1. Hardware diagnostics ───────────────────────────
print("── HARDWARE ─────────────────────────────────────")
try:
    raw = subprocess.check_output(
        ["nvidia-smi",
         "--query-gpu=name,memory.total,memory.free,temperature.gpu",
         "--format=csv,noheader,nounits"],
        text=True
    ).strip().split(", ")
    print(f"  GPU        : {raw[0]}")
    print(f"  VRAM Total : {int(raw[1])/1024:.1f} GB  |  Free: {int(raw[2])/1024:.1f} GB  |  Temp: {raw[3]}°C")
except:
    print("  GPU        : ❌ Not detected — training will be slow on CPU")

ram  = psutil.virtual_memory()
disk = psutil.disk_usage("/content")
print(f"  RAM        : {ram.available/1024**3:.1f} GB free / {ram.total/1024**3:.1f} GB total")
print(f"  Disk       : {disk.free/1024**3:.1f} GB free / {disk.total/1024**3:.1f} GB total")
print(f"  PyTorch    : {torch.__version__}  |  CUDA: {'✅' if torch.cuda.is_available() else '❌ not available'}")

# ── 2. Mount Google Drive ─────────────────────────────
print("\n── GOOGLE DRIVE ─────────────────────────────────")
drive.mount("/content/drive")

for folder in [DRIVE_ANRA_DIR, DRIVE_V2_CHECKPOINTS,
               f"{DRIVE_ANRA_DIR}/identity", f"{DRIVE_ANRA_DIR}/logs",
               f"{DRIVE_ANRA_DIR}/sessions"]:
    os.makedirs(folder, exist_ok=True)

pts = glob.glob(f"{DRIVE_V2_CHECKPOINTS}/*.pt")
dataset_on_drive = os.path.exists(DRIVE_DATASET)
print(f"  Checkpoints: {len(pts)} found {'— will RESUME ✅' if pts else '— will start FRESH'}")
print(f"  Dataset    : {'✅ found' if dataset_on_drive else '❌ MISSING — upload anra_dataset_v6_1.txt to Drive before training'}")
for pt in sorted(pts):
    print(f"    ✅ {os.path.basename(pt)} ({os.path.getsize(pt)/1024**2:.1f} MB)")

# ── 3. Install dependencies ───────────────────────────
print("\n── DEPENDENCIES ─────────────────────────────────")
def pip(pkg):
    return subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                          capture_output=True).returncode == 0

for pkg in ["torch","numpy","sympy","psutil","tqdm","transformers","pytest","sentencepiece"]:
    print(f"  {'✅' if pip(pkg) else '❌'} {pkg}")

MUON_AVAILABLE = False
if pip("muon"):
    try:
        import muon; MUON_AVAILABLE = True
        print("  ✅ muon (optimizer)")
    except ImportError:
        print("  ⚠️  muon install ok but import failed — using AdamW")
else:
    print("  ⚠️  muon unavailable — using AdamW fallback")

print("\n✅ Environment ready")


In [ ]:
# 📤  REPO SETUP
# Upload zip → extract → apply fix → pre-flight check
from google.colab import files
import zipfile, shutil, os, sys, torch, glob

# ── 1. Upload and extract ─────────────────────────────
print("── UPLOAD ───────────────────────────────────────")
print("Upload your An-Ra-the-new-AGI zip when the picker appears...")
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No file uploaded")

zip_name = list(uploaded.keys())[0]
if os.path.exists(REPO_PATH):
    shutil.rmtree(REPO_PATH)
with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall("/content/")

candidates = [d for d in glob.glob("/content/An-Ra*") if os.path.isdir(d) and d != REPO_PATH]
if candidates and not os.path.exists(REPO_PATH):
    os.rename(candidates[0], REPO_PATH)

required = ["training/train_unified.py","training/v2_config.py",
            "training/v2_runtime.py","anra_paths.py","scripts/build_brain.py"]
all_ok = all(os.path.exists(f"{REPO_PATH}/{f}") for f in required)
print(f"  {'✅ Repo extracted and verified' if all_ok else '❌ Some files missing — check your zip'}")

# ── 2. Apply dynamic finetuning fix (idempotent) ──────
print("\n── FIX ──────────────────────────────────────────")

# Fix 1: v2_config.py
cfg_path = f"{REPO_PATH}/training/v2_config.py"
cfg = open(cfg_path).read()
if "unified_trainer_overhead_minutes" in cfg:
    print("  ✅ v2_config.py — already patched")
else:
    old = "    plateau_delta: float = 0.08"
    assert old in cfg, "❌ Cannot find plateau_delta in v2_config.py"
    cfg = cfg.replace(old, old + "\n    unified_trainer_overhead_minutes: int = 5", 1)
    open(cfg_path, "w").write(cfg)
    print("  ✅ v2_config.py — patched")

# Fix 2: train_unified.py
trn_path = f"{REPO_PATH}/training/train_unified.py"
trn = open(trn_path).read()
if "session_timeout_minutes" in trn:
    print("  ✅ train_unified.py — already patched")
else:
    old = ('    identity_cmd = [\n        sys.executable,\n        "-m",\n'
           '        "training.finetune_anra",\n        "--data_path",\n'
           '        str(dataset),\n        "--max_minutes",\n'
           '        str(args.identity_minutes),\n    ]')
    new = ('    session_timeout_minutes = max(1, args.session_minutes - V2_TRAINING.unified_trainer_overhead_minutes)\n'
           '    print(f"[Unified Trainer] Calculated finetuning duration: {session_timeout_minutes} minutes", flush=True)\n\n'
           '    identity_cmd = [\n        sys.executable,\n        "-m",\n'
           '        "training.finetune_anra",\n        "--data_path",\n'
           '        str(dataset),\n        "--max_minutes",\n'
           '        str(session_timeout_minutes),\n    ]')
    assert old in trn, "❌ Cannot find identity_cmd block — check train_unified.py line ~291"
    trn = trn.replace(old, new, 1)
    open(trn_path, "w").write(trn)
    print("  ✅ train_unified.py — patched")

# ── 3. Pre-flight check ───────────────────────────────
print("\n── PRE-FLIGHT ───────────────────────────────────")
sys.path.insert(0, REPO_PATH)
checks = [
    (torch.cuda.is_available(),                     "CUDA available"),
    (os.path.exists(REPO_PATH),                     "Repo present"),
    (os.path.exists(DRIVE_DATASET) or
     os.path.exists(f"{REPO_PATH}/training_data/anra_dataset_v6_1.txt"), "Dataset found"),
    (os.path.exists(DRIVE_V2_CHECKPOINTS),          "Drive checkpoint folder exists"),
    (__import__("importlib").util.find_spec("anra_paths") is not None
     or os.path.exists(f"{REPO_PATH}/anra_paths.py"),  "anra_paths.py present"),
]
RESUME_MODE = bool(glob.glob(f"{DRIVE_V2_CHECKPOINTS}/*.pt"))
checks.append((True, f"Resume mode: {'YES — checkpoints found' if RESUME_MODE else 'NO — starting fresh'}"))

all_ok = True
for ok, label in checks:
    print(f"  {'✅' if ok else '❌'} {label}")
    if not ok: all_ok = False

print(f"\n{'✅ All clear — proceed to training' if all_ok else '❌ Fix issues above before running Cell 4'}")


In [ ]:
# 🚀  RUN TRAINING
import subprocess, sys, os, time

print("╔══════════════════════════════════════════╗")
print("║         AN-RA TRAINING STARTING          ║")
print("╠══════════════════════════════════════════╣")
print(f"║  Mode       : {TRAINING_MODE:<27}║")
print(f"║  Session    : {SESSION_MINUTES} min{' '*(25-len(str(SESSION_MINUTES)))}║")
print(f"║  Ouroboros  : {OUROBOROS_MINUTES} min{' '*(25-len(str(OUROBOROS_MINUTES)))}║")
print(f"║  Identity   : dynamic ({IDENTITY_MINUTES} min budget){' '*(13-len(str(IDENTITY_MINUTES)))}║")
print(f"║  Resume     : {'Yes' if RESUME_MODE else 'No (fresh)':<27}║")
print(f"║  Muon       : {'Yes' if MUON_AVAILABLE else 'No (AdamW fallback)':<27}║")
print("╚══════════════════════════════════════════╝\n")

cmd = [
    sys.executable, "-m", "training.train_unified",
    "--mode",              TRAINING_MODE,
    "--session_minutes",   str(SESSION_MINUTES),
    "--ouroboros_minutes", str(OUROBOROS_MINUTES),
]

env = os.environ.copy()
env["PYTHONPATH"] = REPO_PATH
start = time.time()

process = subprocess.Popen(cmd, cwd=REPO_PATH, env=env,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1)
for line in process.stdout:
    print(line, end="", flush=True)
process.wait()

elapsed = (time.time() - start) / 60
print()
print("╔══════════════════════════════════════════╗")
if process.returncode == 0:
    print(f"║  ✅ Complete — {elapsed:.1f} min{'':>19}║")
else:
    print(f"║  ❌ Failed (code {process.returncode}) — {elapsed:.1f} min{'':>12}║")
print("╚══════════════════════════════════════════╝")
TRAINING_EXIT_CODE = process.returncode


In [ ]:
# ✅  VERIFY DRIVE SAVES
import os, glob

print("── DRIVE CHECKPOINT STATUS ──────────────────────")
pts   = glob.glob(f"{DRIVE_V2_CHECKPOINTS}/*.pt")
jsons = glob.glob(f"{DRIVE_V2_CHECKPOINTS}/*.json")

if pts or jsons:
    for f in sorted(pts + jsons):
        print(f"  ✅ {os.path.basename(f)} ({os.path.getsize(f)/1024**2:.1f} MB)")
else:
    print("  ⚠️  No checkpoint files found in Drive")
    print("  → Training may have failed — check Cell 4 output")

tok = f"/content/drive/MyDrive/AnRa/v2/tokenizer_v2.json"
print(f"\n  Tokenizer  : {'✅ on Drive' if os.path.exists(tok) else '⚠️  not yet saved'}")
print("\n🔁 Next session: run cells 1→4 again. Drive auto-restores.")
